## Pilot verification — run BEFORE each public workshop delivery

This notebook validates the demo-query setup against the **baseline dense** model (which should fail) and the **fine-tuned SPLADE** model (which should pass). It is stricter than the attendee lab: it also checks the manifest, Qdrant collection count, and a reachability sample so stale source machines are caught before cloning.

Run order: every cell, top to bottom. Look for `[FAIL EXPECTED]` on baseline dense and `[PASS]` on fine-tuned SPLADE.


In [ ]:
from pathlib import Path
import sys
import platform
import json
import uuid

if sys.version_info >= (3, 14):
    raise RuntimeError(
        f"Python {platform.python_version()} is not supported for this notebook. "
        "Use Python 3.12; the current FastEmbed/ONNX Runtime stack can "
        "segfault during local embedding on Python 3.14."
    )

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

DATA = REPO / "data"
sys.path.insert(0, str(REPO))

from datasets import load_dataset
from qdrant_client import QdrantClient, models

from retrieval import SpladeEncoder
from scripts.setup_collections import (
    COLLECTION,
    DENSE_VECTOR_NAME,
    SPLADE_VECTOR_NAME,
    DENSE_MODEL as DENSE_MODEL_ID,
    SPLADE_FINETUNED_MODEL_DEFAULT,
)

client = QdrantClient(host="localhost", port=6333)
manifest = json.loads((DATA / "corpus_manifest.json").read_text())
demo_queries = json.loads((DATA / "demo_queries.json").read_text())
splade_encoder = SpladeEncoder(SPLADE_FINETUNED_MODEL_DEFAULT, device="cpu")
assert splade_encoder is not None


def _stable_point_id(product_id: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"esci:{product_id}"))


points_count = client.get_collection(COLLECTION).points_count or 0
expected_count = manifest["corpus_product_count"]
assert points_count == expected_count, (
    f"products count mismatch: Qdrant has {points_count:,}, manifest expects {expected_count:,}"
)

sample_pids = manifest["corpus_sample_product_ids"]
found = client.retrieve(
    collection_name=COLLECTION,
    ids=[_stable_point_id(pid) for pid in sample_pids],
    with_payload=False,
)
assert len(found) == len(sample_pids), "manifest reachability sample failed"


def _normalize_grade(label):
    return str(label)[0].upper() if label else "I"


demo_qids = {int(h["esci_qid"]) for h in demo_queries}
manifest_qids = set(map(int, manifest["eval_query_ids"]))
missing_demos = demo_qids - manifest_qids
assert not missing_demos, f"demo qids missing from corpus manifest: {sorted(missing_demos)}"

_ds = load_dataset(manifest["dataset_id"], split=manifest["split"])
manifest_small_version = manifest.get("small_version")
qrels_by_qid = {qid: {} for qid in demo_qids}
for row in _ds:
    if str(row.get("product_locale", "us")).lower() != str(manifest["locale"]).lower():
        continue
    if manifest_small_version is not None:
        try:
            if int(row.get("small_version", manifest_small_version)) != int(manifest_small_version):
                continue
        except (TypeError, ValueError):
            pass
    try:
        qid = int(row["query_id"])
    except (KeyError, TypeError, ValueError):
        continue
    if qid in qrels_by_qid:
        qrels_by_qid[qid][row["product_id"]] = _normalize_grade(row.get("esci_label"))

for h in demo_queries:
    h["qrels"] = qrels_by_qid[int(h["esci_qid"])]

print(f"[OK] products collection matches manifest ({points_count:,} points)")
print(f"[OK] reachability spot-check: {len(found)}/{len(sample_pids)} sample products present")
print(f"[OK] loaded live ESCI qrels for {len(demo_queries)} demo queries")


def retrieved_ids(points):
    return [p.payload.get("product_id", p.id) for p in points]


def demo_topk_check(label, search_fn, k=3, expect_pass=False):
    """For each demo, check whether an Exact-grade product is in the top-k."""
    print(f"=== {label} — top-{k} ===")
    for q in demo_queries:
        exact_ids = {pid for pid, grade in q["qrels"].items() if grade == "E"}
        if not exact_ids:
            print(f"[SKIP] {q['id']} — no Exact-grade product in live ESCI qrels")
            continue

        ids = retrieved_ids(search_fn(q["text"])[:k])
        hit_rank = next((rank for rank, pid in enumerate(ids, start=1) if pid in exact_ids), None)

        if hit_rank is None:
            if expect_pass:
                print(f"[FAIL] {q['id']} — Exact not in top {k}; review or replace this demo")
            else:
                print(f"[FAIL EXPECTED] {q['id']} — Exact not in top {k}; good hook candidate")
        else:
            if expect_pass:
                print(f"[PASS] {q['id']} — Exact at rank {hit_rank}")
            else:
                print(f"[PASS] {q['id']} — Exact at rank {hit_rank}; consider swapping this demo")


In [ ]:
# Baseline-dense check: expect [FAIL EXPECTED] on every demo. A [PASS] here
# means the baseline already finds the Exact in the top 3 — the workshop
# hook ("the baseline returns a Substitute at rank 1") will not land.
def search_dense(q):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME, limit=3, with_payload=True,
    ).points


demo_topk_check("baseline_dense", search_dense, k=3, expect_pass=False)


In [ ]:
# Fine-tuned SPLADE check: expect [PASS] on every demo. A [FAIL EXPECTED]
# here means the fine-tuned model didn't recover the Exact and the demo
# needs review before going public.
def search_splade(q):
    idx, vals = splade_encoder.encode([q])[0]
    sv = models.SparseVector(
        indices=list(map(int, idx)), values=list(map(float, vals))
    )
    return client.query_points(
        collection_name=COLLECTION,
        query=sv,
        using=SPLADE_VECTOR_NAME, limit=3, with_payload=True,
    ).points


demo_topk_check("splade_finetuned", search_splade, k=3, expect_pass=True)


### Verdict

- **Baseline dense:** every demo should print `[FAIL EXPECTED]`. If any baseline `[PASS]`s, swap that demo before going public — the hook depends on baseline failing.
- **Fine-tuned SPLADE:** focus demo queries should print `[PASS]`. If any prints `[FAIL]`, the fine-tuned model is not recovering the Exact for that demo — either swap the demo or retrain.
- **`[SKIP]` lines** mean the demo has no Exact-grade product in the live ESCI qrels for the manifest-selected corpus; fix the demo list or rebuild the collection.
